In [2]:
import numpy as np
import os
import torch
from torch.utils.data import DataLoader
from matplotlib import pyplot as plt
from accelerate import Accelerator
from torch_ema import ExponentialMovingAverage as EMA
from torchvision import transforms as tf
import torch.distributed as dist

from wavediffusion.model_unet import myUnet
from wavediffusion.model import Scaled
from wavediffusion.wavedata import npyDataWndHist
from wavediffusion.diffusion import ScheduleLogLinear, ScheduleDDPM, samples, masked_training_loop_lp

from wavediffusion.waveutils import evaluate, sample_and_save_lp

%load_ext autoreload
%autoreload 2

In [3]:
train_file_path = '/global/homes/j/jiarongw/scratch_folder/wave_data/train_global/'
stats_file = os.path.join(train_file_path, 'stats.npz')
stats = np.load(stats_file)
meanx, stdx = stats['meanx'], stats['stdx']
meanf, stdf = stats['meanf'], stats['stdf']
test_file_path = '/global/homes/j/jiarongw/scratch_folder/wave_data/test_global/'
test_file_names = [('wave_200804', 'forcing_200804')]
test_file_list = [(os.path.join(test_file_path, f'{x}.npy'), 
                   os.path.join(test_file_path, f'{f}.npy')) for x, f in test_file_names]
test = npyDataWndHist(
    test_file_list,
    resize_x=(320,320), resize_f=(320,320), 
    landmaskname=os.path.join(test_file_path, 'mask.npy'),
    use_icymask=True, compute_stats=False,
    meanx=meanx, stdx=stdx, meanf=meanf, stdf=stdf
)

In [4]:
model = Scaled(myUnet)(in_dim=320, in_ch=1, out_ch=1, ch=256, precond_ch=13, 
                       scale=(test.meanx, test.stdx, test.meanf, test.stdf),
                       ch_mult=(1, 2, 2), attn_resolutions=(16,)) 
ema = EMA(model.parameters(), decay=0.999)
schedule_infer = ScheduleLogLinear(sigma_min=0.01, sigma_max=80, N=80)
loader_test = DataLoader(test, batch_size=2, shuffle=True)  # Used for generating samples during training  
a = Accelerator(mixed_precision="fp16", gradient_accumulation_steps=4) 

In [21]:
from wavediffusion.waveutils import plot_sample

@torch.no_grad()
def sample_and_save_lp(
    model,
    ema,
    loader,
    schedule,
    accelerator,
    path,
    sample_batch_size,
    test, # test dataset for inverting normalization
    filename="sample",):   

    loader_iter = iter(loader)
    with ema.average_parameters():
        x, f, mask = next(loader_iter)
        x = x.to(accelerator.device)
        f = f.to(accelerator.device)
        mask = mask.to(accelerator.device)
        
        # Only generating field 1
        *xt, x0 = samples(
            model, schedule.sample_sigmas(80), gam=1, batchsize=sample_batch_size, 
            accelerator=accelerator, cond=f, mask=mask,
        )

        for i in range(sample_batch_size):
            # Ad-hoc fix: use truth for x[0]
            x0_concat = torch.cat([x[i,[0]], x0[i,[0]], x[i,[2]], x[i,[3]]], dim=0)
            x0_ = test.invert_x(x0_concat) * tf.Resize((320,720))(mask[i].to(x0))
            x_  = test.invert_x(x[i])  * tf.Resize((320,720))(mask[i].to(x))
            f_  = test.invert_f(f[i])

            fig = plot_sample(
                x_.cpu().numpy()[[0], ::-1],
                x0_.unsqueeze(0).cpu().numpy()[:, [0], ::-1],
                f_.cpu().numpy()[:, ::-1],
            )
            fig.savefig(path + filename + f"_{i}.png")
            plt.close(fig)

In [6]:
ema.to(a.device)
model.to(a.device)

myUnetScaled(
  (sig_embed): SigmaEmbedderSinCos(
    (mlp): Sequential(
      (0): Linear(in_features=2, out_features=1024, bias=True)
      (1): SiLU()
      (2): Linear(in_features=1024, out_features=1024, bias=True)
    )
  )
  (conv_in): Conv2d(14, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (downs): ModuleList(
    (0): Module(
      (blocks): ModuleList(
        (0-1): 2 x CondSequential(
          (0): ResnetBlock(
            (layer1): Sequential(
              (0): GroupNorm(32, 256, eps=1e-06, affine=True)
              (1): SiLU()
              (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            )
            (temb_proj): Sequential(
              (0): SiLU()
              (1): Linear(in_features=1024, out_features=256, bias=True)
            )
            (layer2): Sequential(
              (0): GroupNorm(32, 256, eps=1e-06, affine=True)
              (1): SiLU()
              (2): Dropout(p=0.1, inplace=False)
              (

In [22]:
sample_and_save_lp(model, ema, loader_test, schedule_infer, a, 
                   '/global/homes/j/jiarongw/scratch_folder/log1p/lp_hist2d/', 2, 
                   test=test, filename=f"sample_epoch")

torch.Size([2, 4, 320, 320])
torch.Size([2, 1, 320, 320])
torch.Size([4, 320, 320])
torch.Size([4, 320, 320])
